# LangGraph

将Agent执行流程建模成一种状态机State Machine，并将其表示为有向图Directed Graph。图的节点Nodes代表一个具体的计算步骤（调用LLM、执行工具），而边Edges则定义了一个节点到另一个节点的跳转逻辑。天然支持循环

- 全局状态 State：整个执行过程都围绕一个共享状态对象进行。通常定义成 TypedDict 对象，包含任何需要追踪的信息（如对话历史、中间结果、迭代次数等），所有节点都能读取和更行这个状态。
```
# 定义全局状态的数据结构
class AgentState(TypedDict):
    messages: List[str]  # 对话历史
    current_task: str   # 当前任务
    final_answer: str   # 最终答案
    # ...任何其他需要追踪的状态信息
```

- 节点 Nodes：接收当前状态作为输入，返回一个更新后的状态作为输出的python函数。是执行具体工作的单元。
```
# 定义一个“规划者”节点函数
def planer_node(state: AgentState) -> AgentState:
    """根据当前任务制定计划，并更新状态"""
    current_task = state["current_task"]
    # ... 调用LLM生成计划 ...
    plan = f"为任务'{current_task}'生成的计划..."

    # 将新消息追加到状态中
    state["messages"].append(plan)
    return state

# 定义一个“执行者”节点函数
def executor_node(state: AgentState) -> AgentState:
    """执行最新计划，并更新状态"""
    latest_plan = state["messages"][-1]
    # ... 执行计划并获得结果 ...
    result = f"执行计划 '{latest_plan}' 的结果..."

    state["messages"].append(result)
    return state
```

- 边 Edges：连接节点，定义工作流的方向。常规边，指定了一个节点的输出总是流向另一个固定的节点。条件边(conditional edges)，通过一个函数判断当前状态，然后动态地决定下一步应该跳转到哪个节点。
```
def should_continue(state: AgentState) -> str:
    """条件函数：根据状态决定下一步路由"""
    # 假设 如果消息少于3条，则需要继续提规划
    if len(state['messages']) < 3:
        # 返回的字符串需要与添加条件边时定义的键匹配
        return "continue_to_planner"
    else:
        state["final_answer"] = state["messages"][-1]
        return "end_workflow"
```

- 组装工作流
```
from langgraph.graph import StateGraph, END

# 初始化一个状态图，并绑定定义的状态结构
workflow = StateGraph(AgentState)

# 添加节点函数
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)

# 设置图的入口点
workflow.set_entry_point("planner")

# 添加常规边，连接planner和executor
workflow.add_edge("planner", "executor")

# 添加条件边，实现动态路由
workflow.add_conditional_edges(
    # 起始节点
    "executor",
    # 判断函数
    should_continue,
    # 路由映射：将判断函数的返回值映射到目标节点
    {
        "continue_to_planner": "planner",   # 如果返回"continue_to_planner"，则跳回"planner"
        "end_workflow": END                 # 如果返回"end_workflow"，则结束流程
    }
)

# 编译图，生成可执行的应用
app = workflow.compile()

# 运行图
inputs = {"current_task": "分析最近的ai行业新闻", "messages": []}
for event in app.stream(inputs):
    print(event)
```

In [3]:
import os
import sys
from getpass import fallback_getpass

sys.path.append(r"C:\Users\61075\PycharmProjects\learning\hello_agent")
from _LLM import AgentsLLM
from typing import TypedDict, Annotated
import asyncio
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient # 用于agent联网搜索
from dotenv import load_dotenv

load_dotenv(r"C:\Users\61075\PycharmProjects\learning\hello_agent\.env")

True

利用LangGraph搭建问答对话助手：
- understand：分析用户的查询意图
- search：模拟搜索与意图相关的信息
- answer：基于意图和搜索到的信息生成最终答案

定义全局状态

In [2]:
class SearchState(TypedDict):
    messages: Annotated[list, add_messages]
    user_query: str     # 经过LLM理解后的用户需求总结
    search_query: str   # 优化后用于Tavily API的搜索查询
    search_results:str  # Tavily搜索返回的结果
    final_answer:str    # 最终生成的答案
    step: str           # 标记当前步骤

定义工作流节点

In [5]:
# 初始化语言模型
llm = ChatOpenAI(
    model = os.getenv("LLM_MODEL_ID"),
    api_key = os.getenv("LLM_API_KEY"),
    base_url = os.getenv("LLM_BASE_URL"),
    temperature = 0.7
)

# 初始化 tavily 客户端
tavily_client = TavilyClient(
    api_key = os.getenv("TAVILY_API_KEY"),
)

# 1.理解和查询节点
def understand_query_node(state: SearchState) -> dict:
    """步骤1，理解用户查询并生成搜索关键词"""
    user_message = state["messages"][-1].content

    understand_prompt = f"""分析用户的查询："{user_message}"
请完成两个任务：
1. 简洁总结用户想要了解什么
2. 生成最适合搜索引擎的关键词（中英文均可，要精准）

格式：
理解：[用户需求总结]
搜索词：[最佳搜索关键词]"""

    response = llm.invoke([SystemMessage(content=understand_prompt)])
    response_text = response.content

    # 解析LLM的输出，提取搜索关键词
    search_query = user_message     # 默认使用原始查询
    if "搜索词" in response_text:
        search_query = response_text.split("搜索词：")[1].strip()
    elif "搜索关键词" in response_text:
        search_query = response_text.split("搜索关键词：")[1].strip()

    return {
        "user_query": response_text,
        "search_query": search_query,
        "step": "understood",
        "messages": [AIMessage(content=f"我将为你搜索：{search_query}")],
    }

# 2.搜索节点
def tavily_search_node(state: SearchState) -> dict:
    """步骤2：使用 tavily api 进行搜索"""
    search_query = state["search_query"]
    try:
        print(f"%%% 正在搜索：{search_query}")
        response = tavily_client.search(
            query=search_query,
            search_depth="basic",
            max_results=5,
            include_answer=False,    # 是否包含AI生成的回答
        )

        # 处理搜索结果
        search_results = ""

        if response.get("answer"):  # 优先使用tavily的综合答案
            search_results = f"综合答案：\n{response['answer']}\n\n"

        if response.get("results"): # 添加具体的搜索结果
            search_results += "相关信息：\n"
            for i, result in enumerate(response["results"][:3], 1): # 最多取前三条，编号从1开始
                title = result.get("title", "")
                content = result.get("content", "")
                url = result.get("url", "")
                search_results += f"{i}. {title}\n{content}\n来源：{url}\n\n"

        if not search_results:
            search_results = "抱歉，没有找到相关信息。"

        return {
            "search_results": search_results,
            "step": "searched",
            "messages": [AIMessage(content="搜索完成，找到了相关信息，正在整理答案。。。")],
        }

    except Exception as e:
        print(e)

        return {
            "search_results": f"搜索失败：{e}",
            "step": "search_failed",
            "messages": [AIMessage(content="搜索遇到问题，将基于已有知识回答。")]
        }

# 3.回答节点
def generate_answer_node(state: SearchState) -> dict:
    """步骤3.基于搜索结果生成最终答案"""
    if state['step'] == "search_failed":
        # 如果搜索失败，执行回退策略，基于LLM自身知识回答
        fallback_prompt = f"""
搜索API暂时不可用，请基于你的知识回答用户的问题：
用户问题：{state['user_query']}
请提供一个有用的回答，并说明这是基于已有知识的回答。"""
        response = llm.invoke([SystemMessage(content=fallback_prompt)])
        return {
            "final_answer": response.content,
            "step": "completed",
            "messages": [AIMessage(content=response.content)],
        }
    else:
        # 搜索成功，基于搜索结果生成答案
        answer_prompt = f"""基于以下搜索结果为用户提供完整、准确的答案：
用户问题：{state['user_query']}
搜索结果：\n{state['search_results']}
请综合搜索结果，提供准确、有用的回答。
如果是技术问题，提供具体的解决方案或代码。
引用重要信息的来源。
回答结构清晰、易于理解。
如果搜索结果不够完整，请说明并提供补充建议。"""
        response = llm.invoke([SystemMessage(content=answer_prompt)])
        return {
            "final_answer": response.content,
            "step": "completed",
            "messages": [AIMessage(content=response.content)],
        }

构建图，将所有节点连接起来

In [15]:
def create_search_assistant():
    workflow = StateGraph(SearchState)

    # 添加三个节点
    workflow.add_node("understand", understand_query_node)
    workflow.add_node("search", tavily_search_node)
    workflow.add_node("answer", generate_answer_node)

    # 设置线性流程
    workflow.add_edge(START, "understand")
    workflow.add_edge("understand", "search")
    workflow.add_edge("search", "answer")
    workflow.add_edge("answer", END)

    # 编译图
    memory = InMemorySaver()
    app = workflow.compile(checkpointer=memory)
    return app

In [16]:
async def main():
    """主函数：运行智能搜索助手"""

    # 检查API密钥
    if not os.getenv("TAVILY_API_KEY"):
        print("❌ 错误：请在.env文件中配置TAVILY_API_KEY")
        return

    app = create_search_assistant()

    print("🔍 智能搜索助手启动！")
    print("我会使用Tavily API为您搜索最新、最准确的信息")
    print("支持各种问题：新闻、技术、知识问答等")
    print("(输入 'quit' 退出)\n")

    session_count = 0

    while True:
        user_input = input("🤔 您想了解什么: ").strip()

        if user_input.lower() in ['quit', 'q', '退出', 'exit']:
            print("感谢使用！再见！👋")
            break

        if not user_input:
            continue

        session_count += 1
        config = {"configurable": {"thread_id": f"search-session-{session_count}"}}

        # 初始状态
        initial_state = {
            "messages": [HumanMessage(content=user_input)],
            "user_query": "",
            "search_query": "",
            "search_results": "",
            "final_answer": "",
            "step": "start"
        }

        try:
            print("\n" + "="*60)

            # 执行工作流
            async for output in app.astream(initial_state, config=config):
                for node_name, node_output in output.items():
                    if "messages" in node_output and node_output["messages"]:
                        latest_message = node_output["messages"][-1]
                        if isinstance(latest_message, AIMessage):
                            if node_name == "understand":
                                print(f"🧠 理解阶段: {latest_message.content}")
                            elif node_name == "search":
                                print(f"🔍 搜索阶段: {latest_message.content}")
                            elif node_name == "answer":
                                print(f"\n💡 最终回答:\n{latest_message.content}")

            print("\n" + "="*60 + "\n")

        except Exception as e:
            print(f"❌ 发生错误: {e}")
            print("请重新输入您的问题。\n")


In [17]:
await main()

🔍 智能搜索助手启动！
我会使用Tavily API为您搜索最新、最准确的信息
支持各种问题：新闻、技术、知识问答等
(输入 'quit' 退出)


🧠 理解阶段: 我将为你搜索：泰安五月天气、泰山五月份天气预报、5月2-3日泰安天气预报
%%% 正在搜索：泰安五月天气、泰山五月份天气预报、5月2-3日泰安天气预报
🔍 搜索阶段: 搜索完成，找到了相关信息，正在整理答案。。。

💡 最终回答:
根据搜索结果，5月2号和3号泰安的天气情况如下：

1. 温度：根据来源1，五月份泰安市的平均温度为19.7°C，最高温度约为25.5°C，lowest温度约为14°C。来源2指出，五月份泰安的平均温度是13℃ - 26℃，白天平均26℃，夜间平均13℃。
2. 降水量：来源1指出，五月份泰安市的降水量介于每日总量4.1毫米/英吋与20.6毫米/英吋之间波动。来源3指出，泰安泰山近3年的每年5月份：月均降雨（4.3）天，月均降雨量（17.3）毫米。
3. 空气质量：来源3指出，泰安泰山05月份空气最好的情况是32优，空气最差的情况是95良。

综合这些信息，5月2号和3号泰安的天气预报可能如下：

* 温度：白天温度较高，约为25-26℃，夜间温度较低，约为13-14℃。
* 降水量：可能会有降雨，降水量约为4-20毫米/英吋。
* 空气质量：一般良好，可能会有轻微的空气污染。

来源：
1. https://zh.climate-data.org/%E4%BA%9E%E6%B4%B2/%E4%B8%AD%E5%9B%BD/%E5%B1%B1%E4%B8%9C/%E6%B3%B0%E5%AE%89%E5%B8%82-2566/t/%E4%BA%94%E6%9C%88-5/
2. https://weather.zuzuche.com/c3017_m5.html
3. https://www.tianqi24.com/taishanqu/history05.html

请注意，这些信息仅供参考，实际天气情况可能会有所不同。为了获取更准确的天气预报，建议您查看当地的天气网站或应用程序。


感谢使用！再见！👋
